# Load Tether Analysis XArray Data

This notebook demonstrates how to load and work with tether analysis data exported in XArray format from the PyFMGUI DyNaMo tether analysis tool.

The notebook covers:
1. Loading session data from NetCDF files
2. Loading analysis results from NetCDF files 
3. Basic data exploration and visualization
4. Working with the structured scientific data format

In [27]:
# Import required libraries
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
import json

# Set up plotting
plt.style.use('default')
%matplotlib inline

## 1. Loading Session Data from XArray NetCDF Files

The tether analysis GUI exports session data in NetCDF format using XArray. This includes:
- File paths and names
- Good/bad file classifications  
- Analysis parameters for each file
- Plateau selections for each file
- Rich metadata and attributes

**To get started:**
1. Update the `session_file_path` variable in the next cell with the path to your `*_xarray.nc` file
2. Optionally set `results_file_path` if you have individual analysis results to load

In [28]:
# Define path to your session NetCDF file
# Update this path to point to your actual session file
session_file_path = "/Users/evillz/Data/article/2025_07_01_THP1_phd/sessions/latest/test/tether_session_20250716_184416_xarray.nc"

print(f"Looking for session file at: {session_file_path}")

# Check if file exists
if os.path.exists(session_file_path):
    print("✅ Session file found!")
    
    # Load the session dataset
    session_ds = xr.open_dataset(session_file_path)
    
    print("✅ Session data loaded successfully!")
    print(f"Dataset contains {len(session_ds.file)} files")
    
    # Display basic info about the dataset
    print("\nDataset structure:")
    print(session_ds)
else:
    print("❌ Session file not found!")
    print("Please update the 'session_file_path' variable above with the correct path to your .nc file")
    session_ds = None

Looking for session file at: /Users/evillz/Data/article/2025_07_01_THP1_phd/sessions/latest/test/tether_session_20250716_184416_xarray.nc
✅ Session file found!
✅ Session data loaded successfully!
Dataset contains 272 files

Dataset structure:
<xarray.Dataset> Size: 526kB
Dimensions:             (file: 272)
Coordinates:
  * file                (file) int64 2kB 0 1 2 3 4 5 ... 266 267 268 269 270 271
Data variables:
    file_path           (file) <U118 128kB ...
    file_name           (file) <U58 63kB ...
    bool_good_curve     (file) bool 272B ...
    file_parameters     (file) <U258 281kB ...
    plateau_selections  (file) <U47 51kB ...
Attributes:
    title:                   Tether Analysis Session Data
    description:             Session data from Tether Analysis GUI with per-f...
    creation_timestamp:      20250716_184416
    creation_date:           2025-07-16T18:44:21.830043
    software:                Tether Analysis GUI v2
    total_files:             272
    good_files: 

In [29]:
# Explore the session data structure
if session_ds is not None:
    print("=== SESSION DATA EXPLORATION ===")
    
    # Show dimensions
    print(f"\nDimensions: {dict(session_ds.dims)}")
    
    # Show coordinates
    print(f"\nCoordinates:")
    for coord in session_ds.coords:
        print(f"  {coord}: {session_ds.coords[coord].shape}")
    
    # Show data variables
    print(f"\nData Variables:")
    for var in session_ds.data_vars:
        print(f"  {var}: {session_ds.data_vars[var].shape} - {session_ds.data_vars[var].attrs.get('description', 'No description')}")
    
    # Show global attributes
    print(f"\nGlobal Attributes:")
    for attr, value in session_ds.attrs.items():
        print(f"  {attr}: {value}")
    
    # Show first few file paths
    print(f"\nFirst 5 file paths:")
    for i in range(min(5, len(session_ds.file_path))):
        print(f"  {i}: {session_ds.file_path[i].values}")
else:
    print("Session dataset not loaded. Please check the file path above.")

=== SESSION DATA EXPLORATION ===

Dimensions: {'file': 272}

Coordinates:
  file: (272,)

Data Variables:
  file_path: (272,) - Full local file system paths to TDMS files
  file_name: (272,) - Base names of TDMS files without directory path
  bool_good_curve: (272,) - Boolean flags indicating whether curve was marked as good (True) or bad (False)
  file_parameters: (272,) - JSON-encoded analysis parameters for each file (mixed numeric/string/boolean types)
  plateau_selections: (272,) - JSON-encoded boolean arrays indicating which plateaus were selected for each file

Global Attributes:
  title: Tether Analysis Session Data
  description: Session data from Tether Analysis GUI with per-file parameters and selections
  creation_timestamp: 20250716_184416
  creation_date: 2025-07-16T18:44:21.830043
  software: Tether Analysis GUI v2
  total_files: 272
  good_files: 272
  bad_files: 0
  data_format_version: 2.0
  root_directory: /Users/evillz/Data/article/2025_07_01_THP1_phd
  coordinate_d

/var/folders/kc/jykhdgwx4p3696y51hzh6j3r0000gn/T/ipykernel_69508/3126906594.py:6: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"\nDimensions: {dict(session_ds.dims)}")


In [30]:
# Work with file parameters
if session_ds is not None:
    print("=== WORKING WITH FILE PARAMETERS ===")
    
    # Convert parameters to pandas DataFrame for easier analysis
    param_data = []
    for i in range(len(session_ds.file)):
        params_str = session_ds.file_parameters[i].values
        try:
            # Convert numpy array to string if needed
            if hasattr(params_str, 'item'):
                params_str = params_str.item()
            elif isinstance(params_str, np.ndarray):
                params_str = str(params_str)
            
            params = json.loads(params_str)
            params['file_index'] = i
            params['file_name'] = str(session_ds.file_name[i].values)
            params['bool_good_curve'] = bool(session_ds.bool_good_curve[i].values)
            param_data.append(params)
        except (json.JSONDecodeError, TypeError) as e:
            print(f"Warning: Could not parse parameters for file {i}: {e}")
            print(f"  Data type: {type(params_str)}")
            print(f"  Data: {params_str}")
    
    if param_data:
        param_df = pd.DataFrame(param_data)
        print(f"\nParameters DataFrame shape: {param_df.shape}")
        print("\nParameter columns:", list(param_df.columns))
        
        # Show summary statistics for numeric parameters
        numeric_params = param_df.select_dtypes(include=[np.number])
        print(f"\nSummary of numeric parameters:")
        print(numeric_params.describe())
        
        # Show first few rows
        print(f"\nFirst 3 parameter sets:")
        print(param_df.head(3).to_string())
    else:
        print("No parameter data could be parsed")
        
    # Also show plateau selections summary
    print(f"\n=== PLATEAU SELECTIONS SUMMARY ===")
    plateau_summary = []
    for i in range(len(session_ds.file)):
        try:
            plateau_str = session_ds.plateau_selections[i].values
            # Convert numpy array to string if needed
            if hasattr(plateau_str, 'item'):
                plateau_str = plateau_str.item()
            elif isinstance(plateau_str, np.ndarray):
                plateau_str = str(plateau_str)
                
            plateau_selections = json.loads(plateau_str)
            
            summary = {
                'file_index': i,
                'file_name': str(session_ds.file_name[i].values),
                'total_plateaus': len(plateau_selections),
                'selected_plateaus': sum(plateau_selections),
                'good_curve': bool(session_ds.bool_good_curve[i].values)
            }
            plateau_summary.append(summary)
        except (json.JSONDecodeError, TypeError) as e:
            print(f"Warning: Could not parse plateau selections for file {i}: {e}")
    
    if plateau_summary:
        plateau_df = pd.DataFrame(plateau_summary)
        print(f"\nPlateau selections DataFrame shape: {plateau_df.shape}")
        print("\nPlateau selections summary:")
        print(plateau_df.to_string())
        
        # Summary statistics
        print(f"\n=== PLATEAU STATISTICS (from selections) ===")
        print(f"Total files: {len(plateau_df)}")
        print(f"Good curve files: {plateau_df['good_curve'].sum()}")
        print(f"Files with plateaus: {(plateau_df['total_plateaus'] > 0).sum()}")
        print(f"Average plateaus per file: {plateau_df['total_plateaus'].mean():.1f}")
        print(f"Average selected plateaus per file: {plateau_df['selected_plateaus'].mean():.1f}")
    
else:
    print("Session dataset not loaded. Please check the file path above.")

=== WORKING WITH FILE PARAMETERS ===

Parameters DataFrame shape: (272, 13)

Parameter columns: ['sav_window_length', 'sav_polyorder', 'pl_threshold', 'pl_min_width', 'last_num_plateaus', 'last_plateau_avg_percentage', 'max_offset', 'min_offset', 'z_sensor_delay', 'bool_correct_overshoot', 'file_index', 'file_name', 'bool_good_curve']

Summary of numeric parameters:
       sav_window_length  sav_polyorder  pl_threshold  pl_min_width  \
count         272.000000     272.000000  2.720000e+02    272.000000   
mean           25.511029       2.628676  4.133379e-08      7.360294   
std            14.567310       1.615762  5.272473e-08      8.605476   
min             5.000000       1.000000  1.000000e-09      1.000000   
25%            10.000000       1.000000  1.500000e-08      1.000000   
50%            26.000000       2.000000  1.700000e-08      2.000000   
75%            32.000000       5.000000  2.900000e-08     20.000000   
max            50.000000       5.000000  1.860000e-07     20.00